# Solution E1 · Feature importances with the wrong column names

**Type**: Find the bug

## Objective

Find out why a report says the largest importance belongs to the column `pclass` when the model mostly uses **sex**: the colleague assigned the importances of the pipeline to the names of the original columns, but the `ColumnTransformer` has **reordered the columns** (categoricals first, then numerics).
Fix it with `get_feature_names_out()`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
CAT = ["sex", "port", "deck", "title"]
NUM = ["pclass", "age", "siblings_spouses", "parents_children", "fare"]
prep = ColumnTransformer([("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OrdinalEncoder()), CAT),
                          ("num", SimpleImputer(strategy="median"), NUM)])
pipe = make_pipeline(prep, DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=0)).fit(X_tr, y_tr)

## Commented solution

In [ ]:
imp_bug = pd.Series(pipe[-1].feature_importances_, index=X_tr.columns)          # "the importances, with the column names"
top_bug = imp_bug.idxmax()
print(f"most important feature according to the report: {top_bug} ({imp_bug.max():.2f})")
imp_bug.round(3).sort_values(ascending=False).head(4)

**Where the bug is.** `feature_importances_` has one entry for each column **coming out of the preprocessing**, in the order in which the `ColumnTransformer` produces them: first the 4 categoricals (`sex`, `port`, `deck`, `title`), then the 5 numerics. The colleague labels them with `X_tr.columns`, where the order is different
(`pclass`, `sex`, `age`, ...): the very high importance of **sex** (position 0 in the output) is attributed to `pclass` (position 0 in the input). The lengths match (9 and 9), so there is no error.

In [ ]:
pipeline_names = list(pipe[:-1].get_feature_names_out())
clean = [n.split("__", 1)[1] for n in pipeline_names]
different_positions = sum(a != b for a, b in zip(X_tr.columns, clean))
print("input columns :", list(X_tr.columns))
print("output columns:", clean)
print("different positions:", different_positions, "out of 9")

In [ ]:
imp_ok = pd.Series(pipe[-1].feature_importances_, index=clean)
top_ok = imp_ok.idxmax()
perm = permutation_importance(pipe, X_te, y_te, n_repeats=10, random_state=0, scoring="roc_auc")
top_permutation = pd.Series(perm.importances_mean, index=X_te.columns).idxmax()
print(f"correct importances, top three: {imp_ok.nlargest(3).round(3).to_dict()} | most important: {top_ok} | by permutation: {top_permutation}")

## Check

In [ ]:
with verify():
    check(top_bug == "pclass", "the colleague's report says the most important feature is `pclass`", "don't modify the colleague's code")
    check(pipeline_names is not None and len(pipeline_names) == 9 and pipeline_names[0].startswith("cat__sex") and different_positions >= 6, "the pipeline has 9 columns in a different order (cat before num): at least 6 different positions", "pipe[:-1].get_feature_names_out()")
    check(imp_ok is not None and top_ok == "sex" and imp_ok.max() > 0.4 and not any(str(n).startswith(("cat__", "num__")) for n in imp_ok.index), "the most important feature is `sex` (over 0.4); the names have no prefix", "remove 'cat__' and 'num__' from the names")
    check(close(imp_ok.sum(), 1.0) and set(imp_ok.index) == set(X_tr.columns), "same columns as X, importances that sum to 1", "reindex with the correct names")
    check(top_permutation == top_ok, "the permutation confirms: the most important column is `sex`", "permutation_importance(pipe, X_te, y_te, ...) has the same names as X_te.columns")
done()

## What you learned

- **The importances follow the columns coming out of the preprocessing**, not those going in. `ColumnTransformer` reorders (the blocks in the order in which you declare them), the one-hot multiplies the columns, imputers with indicators add some, selectors remove some. The only correct way to get the names is `get_feature_names_out()` of the pipeline (without the last step).
- The bug is invisible when **the number of columns stays the same**: with the one-hot you would get a length error and fix it; with the ordinal encoding (9 columns in and out) the report comes out wrong and nobody notices. A report with `pclass` at the top is plausible ("the rich survive"), and the wrong story is easy to tell.
- The check is always the same: **compare with a second method** (the permutation importance, which works on the input columns and has the same names as `X`) or with your knowledge of the problem (`sex` and `title` are obvious predictors).
- Every time you stick names on a numeric array coming from a pipeline, ask who decided the order.

## Common pitfall

Using `X.columns` as the index of `feature_importances_`, `coef_` or of the `shap_values` arrays of a model inside a pipeline. If the preprocessing changes the number or the order of the columns, the label is wrong without any warning.

## Interview question

<details><summary>How do you assign names to the importances of a model inside a pipeline?</summary>

With `get_feature_names_out()` of the preprocessing part (`pipeline[:-1]`), because the final model sees columns that are reordered, expanded or removed. Then I check with a second method, like the permutation on the input columns, that the conclusions agree, and I add an automatic test that compares the length and the names.

</details>